# Phase 5 – Synthesis: which lever wins the most heats?

Phases 3–4 produced about 60 result tables, each in its own unit: seconds per run, percentage points of clean heats, expected seconds per heat. The coach needs **one ranking**. This notebook converts every lever into the same currency, **extra heats won per season**, and collects every number the findings and the deck quote.

Everything calls `src/flyball/synthesis.py`; `scripts/run_phase5_synthesis.py` rebuilds `reports/phase5/` in about 25 s. The written deliverables built from these outputs are `reports/findings.md` (findings ranked by impact, with actions) and `reports/storyline.md` (slide-by-slide outline). Every step is logged in `ANALYSIS_LOG.md` Step 6.

In [1]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from flyball import config, metrics, synthesis

warnings.simplefilter("ignore")          # MixedLM convergence chatter (line-up profiles)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
t = metrics.load_processed()

---
## 1. The currency: the heat-win model

A **won heat** is the good result (Phase 3). The Phase 3 WRFT heat-win model gives P(win) from:
- the typical **speed gap** to the opponent (the team's usual clean time that day minus the opponent's);
- the **start gap**;
- **who faulted** in the heat (false start, early pass, run fault; own and opponent's).

A lever is a change to those inputs. For example, "XXL false starts down to its division peers" lowers the share of XXL heats with a false start from 35 % to 15 %, by removing false starts at random among them. The ledger then adds up how much the predicted P(win) rises over WRFT's heats.

**Only heats against other clubs count** (907 decided heats): a club-wide fix can't win WRFT a heat against itself. The CIs come from 2,000 draws of the model coefficients and, where the size of the lever is itself an estimate (a gap to the peers, the day-2 effect, the heat slope), of that size too.

In [2]:
frame = synthesis.ledger_frame(t)
fit = frame["fit"]
coefs = pd.DataFrame({"odds_ratio": np.exp(fit.params), "ci_low": np.exp(fit.conf_int()[0]),
                      "ci_high": np.exp(fit.conf_int()[1])}).round(3)
display(coefs)
print(f"Decided heats against other clubs: {frame['n_decided']}, WRFT won {frame['win_rate']:.1%}; "
      f"in the model: {len(frame['d'])} (the rest lack a typical speed on one side)")

,odds_ratio,ci_low,ci_high
Intercept,1.023,0.838,1.250
speed_gap,0.668,0.571,0.781
start_gap,0.151,0.009,2.528
own_false_start,0.132,0.063,0.280
own_pass_fault,0.108,0.065,0.182
own_run_fault,0.077,0.027,0.216
opp_false_start,2.928,1.761,4.870
opp_pass_fault,3.818,2.543,5.732
opp_run_fault,3.493,1.393,8.758


Decided heats against other clubs: 907, WRFT won 39.0%; in the model: 897 (the rest lack a typical speed on one side)


- An own false start or early pass multiplies the odds of winning by about 0.1–0.13, and a run fault by about 0.08. An opponent's fault multiplies them by about 3–4.
- Each second of typical speed disadvantage multiplies the odds by 0.67.
- So a fault is worth roughly 5 to 6½ seconds of typical speed. That is why the fault levers rank high.

---
## 2. A hand check of the engine

The ledger averages over every combination of the three fault types analytically. Here is one lever done by hand: XXL's false starts down to its division peers. Each of XXL's false-start heats loses its false start with probability 1 − target/current, and the gain is the change in the model's P(win).

In [3]:
d = frame["d"]
peer = metrics.peer_benchmark(t["heats"])
gap = peer.loc[peer["team"].eq("WildRunners XXL") & peer["metric"].eq("false_start_rate"), "estimate"].iat[0]
xxl = d[d["team_norm"].eq("wildrunners xxl")]
current = xxl["own_false_start"].mean()
target = current - gap
faulted = xxl[xxl["own_false_start"].eq(1)]
gain = (1 - target / current) * (fit.predict(faulted.assign(own_false_start=0)) - fit.predict(faulted))
print(f"XXL false-start heats: {current:.1%} now, {target:.1%} at the peers' level")
print(f"Extra wins by hand: {gain.sum() * frame['scale']:.2f}")

XXL false-start heats: 35.4% now, 15.4% at the peers' level
Extra wins by hand: 3.63


The ledger below gives the same 3.63 extra wins for this lever. The day-2 lever was also checked by Monte Carlo (400 simulated seasons): 28.2 against 28.2.

---
## 3. The impact ledger

In [4]:
out = synthesis.impact_ledger(t)
ledger = out["impact_ledger"]
cols = ["rank", "group", "lever", "extra_wins", "ci_low", "ci_high", "club_win_rate_pp", "heats_affected"]
display(ledger[cols].round(2))

,rank,group,lever,extra_wins,ci_low,ci_high,club_win_rate_pp,heats_affected
0,1,reference,No own faults at all (ceiling),148.44,127.43,165.37,16.37,907.00
1,2,line-up: bench swap,Zephyr: Rex > Rocky > Scotty > Gordon -> Rocky > Scotty > Rex > Pax,45.03,30.93,57.61,4.96,184.03
2,3,second-day effect,Saturday (day 1) as clean as Sunday (day 2),28.22,17.37,41.53,3.11,468.16
3,4,line-up: reorder (club),All teams run their recommended order (same dogs),27.37,20.30,32.61,3.02,907.00
4,5,fault gap to division peers (club),All teams: every fault gap closed,26.33,20.59,32.76,2.90,907.00
5,6,line-up: bench swap,Hopefuls: Cookies > Enola > Maggie > Tyna -> Maggie > Tyna > Pax > Enola,15.86,12.22,18.58,1.75,139.54
6,7,line-up: bench swap,XXL: Rex > Hank > Gerry > Bailey -> Pax > Hank > Bailey > Gerry,13.10,9.58,16.08,1.44,97.07
7,8,line-up: reorder,Hopefuls: Cookies > Enola > Maggie > Tyna -> Maggie > Tyna > Cookies > Enola,12.15,8.90,14.50,1.34,139.54
8,9,fault gap to division peers (club),All teams: false starts down to peers,11.54,7.74,15.63,1.27,907.00
9,10,line-up: bench swap,Wonders: Rex > Hank > Hector > Gordon -> Pax > Hank > Rex > Hector,8.68,5.79,11.32,0.96,38.42


**How to read it.** `extra_wins` is per season, for a season like 2023 (907 decided heats against other clubs, about 45 per competition day). `club_win_rate_pp` is the same thing as percentage points of WRFT's win rate (39.0 % against other clubs in 2023).

**The levers, by group:**
- **Reference: no own faults at all, +148 wins (+16 pp).** Nobody runs fault-free; this is the ceiling that shows how much the faults cost.
- **Second-day effect, +28 wins.** WRFT's heats on day 1 of a two-day event have 11.6 pp more false starts and 8.2 pp more early passes than the same teams on day 2 (table below). If Saturdays ran like Sundays, that is about 3 pp of win rate. The cause is unknown, so this is a lever to *investigate*, not one to pull directly.
- **Line-up order, +27 wins (all teams together).** Same dogs, better order: Hopefuls +12 (stop opening with Cookies), Zephyr +8, XXL +4, Maximini +3. **Free and available now.**
- **Fault gaps to the division peers, +26 wins** if every team closed every gap: false starts +12, early passes +8, run faults +6. The largest single gaps are Zephyr's run faults (+4.4) and XXL's false starts (+3.6). This needs training.
- **Bench swaps (bring in Pax), +3 to +45 wins.** These rows are *not* additive: Pax can join only one team, and his availability is unknown (last raced in July). The large values (Zephyr, Wonders, XXL) come from replacing the 15 cm height dog Gordon or a slow starter. A team that much faster would be seeded into a faster division, so these are optimistic.
- **Heat, +3.5 wins at most.** Heat slows *every* team, so it changes results only if WRFT copes better than its opponents. Even if WRFT kept its full speed on every hot day, the gain would be small. Heat is a welfare issue, not a results lever.
- **Lane colour and partner sex/neuter status: 0** (Q1 and Q2 found no effect, so they are not levers).

**Levers in different groups overlap.** For example, the Hopefuls reorder *is* a false-start fix (Cookies false-starts 37 % of the time). Rows within a group add up; rows across groups don't.

In [5]:
display(out["impact_day2_faults"].round(4))

,fault,estimate,se,ci_low,ci_high,p_value
0,own_false_start,-0.1158,0.0137,-0.1446,-0.0871,0.0000
1,own_pass_fault,-0.0819,0.0306,-0.1459,-0.0180,0.0148
2,own_run_fault,-0.0012,0.0182,-0.0394,0.0369,0.9475


The day-2 fit: the same WRFT team at the same event, day 2 minus day 1, as a share of heats with each fault. It controls for weather, hurdle and the heat number of the day, and is clustered on venue-day. False starts −11.6 pp, early passes −8.2 pp, run faults no change. It agrees with the dog-level Phase 4B result (false starts −10 pp per start).

---
## 4. Key numbers: one source for the findings and the deck

Every number that `reports/findings.md` or a slide quotes is in `reports/phase5/key_numbers.json`, with its unit, CI, source table and log step. Phase 6 reads the slide numbers from this file, so the deck can't drift from the analysis. A test checks that the file is up to date and that every entry resolves to exactly one row of its table.

In [6]:
keys = json.loads((config.REPORTS / "phase5" / "key_numbers.json").read_text(encoding="utf-8"))
kn = pd.DataFrame.from_dict(keys, orient="index")
display(kn[["value", "ci", "unit", "log_step", "source"]])

,value,ci,unit,log_step,source
wrft_heats_with_result,1116,NaN,count,2.5,reports/phase3/team_summary.csv
wrft_win_rate,0.4077,"[0.3792, 0.4368]",share,2.5,reports/phase3/team_summary.csv
field_win_rate,0.4562,NaN,share,2.5,reports/phase3/team_summary.csv
wrft_clean_heat_rate,0.5439,NaN,share,2.5,reports/phase3/team_summary.csv
field_clean_heat_rate,0.5657,NaN,share,2.5,reports/phase3/team_summary.csv
...,...,...,...,...,...
impact_zephyr_swap,45.0261,"[30.9303, 57.6067]",wins,6.1,reports/phase5/impact_ledger.csv
impact_hopefuls_swap,15.8618,"[12.2154, 18.5768]",wins,6.1,reports/phase5/impact_ledger.csv
impact_heat,3.4528,"[1.7763, 5.3522]",wins,6.1,reports/phase5/impact_ledger.csv
wrft_heats_vs_clubs,907,NaN,count,6.1,reports/phase5/impact_meta.csv


---
## 5. Summary

| Rank | Lever | Extra heats won per season (95 % CI) | Kind |
|---|---|---|---|
| – | No own faults at all | 148 (127–165) | ceiling, for scale |
| 1 | Saturday as clean as Sunday | 28 (17–42) | investigate the cause |
| 2 | Recommended running orders, same dogs | 27 (20–33) | **free, now** |
| 3 | Close every fault gap to the division peers | 26 (21–33) | training |
| – | Bring in Pax (one team only) | 3–45 | depends on availability; optimistic |
| 4 | Keep full speed on hot days (upper bound) | 3.5 (1.8–5.4) | welfare first |
| – | Lane colour, partner sex/neuter status | 0 | no effect found |

These are what-ifs from an observational model. They rank the levers; they are not promises. The findings, the actions and the confidence of each are in `reports/findings.md`.